# Find Optimal K for NMF
Find the optimal number of topics (K) for NMF using **Reconstruction Error** and **Topic Coherence (Cᵥ)** with spaCy preprocessing.

## 1. Installation
Run this cell once to install all required dependencies. You can skip it if they are already installed.

In [ ]:
import sys

!{sys.executable} -m pip install \
    notebook \
    ipywidgets \
    numpy \
    matplotlib \
    tqdm \
    scikit-learn \
    spacy \
    gensim

# Download the spaCy English model
!{sys.executable} -m spacy download en_core_web_sm

## 2. Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
import spacy
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from google.colab import drive

drive.mount('/content/drive')

## 3. Configuration
Set your input file path and K search range here.

In [2]:
INPUT_FILE = '/content/drive/MyDrive/ColabData/math_cs_mixed_s1.json' # Path to your JSON dataset file
START_K    = 2 # Starting value for K (min 2)
END_K      = 40 # Ending value for K
STEP_K     = 1 # Step size for K
OUTPUT_DIR = '/content/drive/MyDrive/ColabData/' # Directory to save the output plot

## 4. Load Data

In [ ]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"File '{INPUT_FILE}' does not exist.")

print(f"Loading data from {INPUT_FILE}...")
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

texts = [item.get('text', '') for item in data if item.get('text', '').strip()]

if not texts:
    raise ValueError("No text data found in the dataset.")

print(f"Loaded {len(texts)} documents.")

## 5. spaCy Setup & Custom Stop Words

In [ ]:
print("Loading spaCy 'en_core_web_sm'...")
# If not installed, run: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

academic_stopwords = [
    'finding', 'findings', 'illustrate', 'significant', 'provide', 'provides', 'potential', 'associated', 'effective', 'aspect', 'aspects', 'challenge', 'challenges',
    'paper', 'study', 'research', 'result', 'results', 'method', 'methodology',
    'proposed', 'propose', 'approach', 'based', 'using', 'used', 'use', 'to', 'we', 'source',
    'analysis', 'model', 'system', 'data', 'application', 'new', 'development',
    'performance', 'conclusion', 'abstract', 'introduction', 'work', 'time',
    'significant', 'shown', 'show', 'demonstrate', 'experiment', 'experimental',
    'university', 'department', 'author', 'et', 'al', 'figure', 'table',
    'high', 'low', 'large', 'small', 'different', 'various', 'property', 'properties', 'increase', 'effect', 'activity',
    'structure', 'compound', 'condition', 'quality', 'entry', 'contain', 'parameter', 'observe', 'report', 'present', 'evaluate'
]

for word in academic_stopwords:
    nlp.vocab[word].is_stop = True

print(f"Added {len(academic_stopwords)} academic stop words.")

custom_stopwords = list(set(list(ENGLISH_STOP_WORDS) + academic_stopwords))

def spacy_tokenizer(text):
    if not text:
        return []
    doc = nlp(text)
    return [
        token.lemma_.lower() for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.like_num
        and len(token) > 2
    ]

## 6. Vectorize with TF-IDF (sklearn)

In [ ]:
print("Fitting TF-IDF Vectorizer with spaCy tokenizer...")
vectorizer = TfidfVectorizer(stop_words=custom_stopwords ,tokenizer=spacy_tokenizer)
tfidf_matrix = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabulary size      : {len(feature_names)}")
print(f"TF-IDF matrix shape  : {tfidf_matrix.shape}")

## 7. Tokenize Texts for Gensim Coherence Model

In [ ]:
print("Tokenizing texts for Gensim (this may take a while)...")
tokenized_texts = [spacy_tokenizer(text) for text in tqdm(texts, desc="Tokenizing")]
dictionary = Dictionary(tokenized_texts)
print(f"Gensim dictionary size: {len(dictionary)}")

## 8. Grid Search over K

In [ ]:
start_k = max(2, START_K)
k_values = list(range(start_k, END_K + 1, STEP_K))
reconstruction_errors = []
coherence_scores      = []

print(f"Starting grid search for K = {k_values}")

for k in tqdm(k_values, desc="Tuning NMF"):
    # Fit NMF
    nmf_model = NMF(n_components=k, random_state=42, max_iter=500)
    nmf_model.fit(tfidf_matrix)

    # Reconstruction Error
    reconstruction_errors.append(nmf_model.reconstruction_err_)

    # Top-10 words per topic
    top_words_per_topic = []
    for topic in nmf_model.components_:
        top_features_ind = topic.argsort()[:-10 - 1:-1]
        top_words_per_topic.append([feature_names[i] for i in top_features_ind])

    # Topic Coherence (Cᵥ)
    cm = CoherenceModel(
        topics=top_words_per_topic,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence_scores.append(cm.get_coherence())

print("Grid search complete.")

## 9. Results Summary

In [ ]:
best_coherence_k = k_values[int(np.argmax(coherence_scores))]
best_recon_k     = k_values[int(np.argmin(reconstruction_errors))]

print(f"{'K':>5} | {'Recon. Error':>14} | {'Coherence (Cᵥ)':>15}")
print("-" * 40)
for k, r, c in zip(k_values, reconstruction_errors, coherence_scores):
    print(f"{k:>5} | {r:>14.4f} | {c:>15.4f}")

print(f"\nBest K by Coherence          : {best_coherence_k}")
print(f"Best K by Reconstruction Error: {best_recon_k}")

## 10. Plot: Reconstruction Error vs Topic Coherence

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Reconstruction Error (left axis)
color_recon = 'tab:red'
ax1.set_xlabel('Number of Topics (K)')
ax1.set_ylabel('Reconstruction Error', color=color_recon)
ax1.plot(k_values, reconstruction_errors, marker='x', color=color_recon,
         linestyle='--', linewidth=2, label='Reconstruction Error')
ax1.tick_params(axis='y', labelcolor=color_recon)
ax1.set_xticks(k_values)
ax1.grid(True, linestyle=':', alpha=0.6)

# Coherence (right axis)
color_coh = 'tab:blue'
ax2 = ax1.twinx()
ax2.set_ylabel('Topic Coherence ($C_v$)', color=color_coh)
ax2.plot(k_values, coherence_scores, marker='o', color=color_coh,
         linewidth=2, label='Topic Coherence')
ax2.tick_params(axis='y', labelcolor=color_coh)

plt.title('Optimal K Selection for NMF (spaCy Preprocessed)\n(Reconstruction Error vs Topic Coherence)')
fig.tight_layout()

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right')

output_path = os.path.join(OUTPUT_DIR, 'nmf_optimal_k_selection.png')
plt.savefig(output_path, dpi=300)
plt.show()

print(f"Plot saved to: {output_path}")